In [1]:
import polars as pl
import json

abb = pl.read_csv("abbreviations.csv")
abb = abb.filter(pl.col("abbreviation").is_not_null())

with open("eindeutigeAbk.json", "r", encoding="utf-8") as f:
    data = json.load(f)

lotta_abb = pl.DataFrame(list(data.values())).rename({"Abkuerzung": "abbreviation", "Ausformulierung": "full_word"})
lotta_abb = lotta_abb.join(abb, on="abbreviation", how="left")

left_out = abb.join(lotta_abb, on="abbreviation", how="anti")

In [ ]:
left_out

In [ ]:
left_out.filter(pl.all_horizontal(pl.col("RG1", "RG2", "RG3", "RG4", "RG5", "RG6", "RG7", "RG8", "RG9", "note", "note1", "note2").is_null()))

In [ ]:
left_out.filter(pl.all_horizontal(pl.col("RG1", "RG2", "RG3", "RG4", "RG5", "RG6", "RG7", "RG8", "RG9").is_null()))

In [ ]:
left_out.filter(pl.all_horizontal(pl.col("note1", "note2").is_null()))

In [ ]:
left_out.filter(pl.col("abbreviation").is_null())

# expansion dictionaries

## for all volumes

This one will only keep entries where the expansion is the same for all volumes. It should/could be complemented with one dictionary per volume with the volume-specific expansions.

things to remove

In [28]:
# (fl.) or (fl. vel duc.)
#lotta_abb.filter(pl.col("full_word").str.contains(r"\(fl\.( vel duc.)?\)"))
# (Plural)
lotta_abb.filter(pl.col("full_word").str.contains(r"\(Plural\)"))

abbreviation,full_word,full_word_right,translation,note,word_stem,declension,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,note1,note2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""appl.""","""apostoli (Plural)""","""apostoli (Plural)""","""die Apostel""",null,"""apostol -orum""","""o""",null,null,null,"""appl.""","""appl.""","""appl.""","""appl.""","""appl.""","""appl.""",null,"""in den Bd. 1, 2, 3 wahrscheinl…"
"""bb.""","""beati (Plural)""","""beati (Plural)""","""die heiligen""",null,"""beat -orum/arum""","""o/a (Adj.)""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""",null,null
"""dd.""","""dicti (Plural)""","""dicti (Plural)""","""die genannten""",null,"""dict -orum""","""o""","""dicti""","""dicti""","""dicti""","""dd.; dicti""","""dicti""","""dd.""","""dd.; dicti""","""dd.; dicti""","""dd.; dicti""",null,null
"""SS.""","""sancti (Plural)""","""sancti (Plural)""","""die heiligen""","""siehe ss.""",null,null,null,null,null,null,null,null,null,null,null,null,null
"""ss.""","""sancti (Plural)""","""sancti (Plural)""","""die heiligen""",null,"""sanct -orum""","""o""","""ss.; SS.""","""ss.""","""ss.""","""ss.""","""ss.""","""ss.""","""ss.""","""ss.""","""ss.""",null,null


entries to ignore

In [12]:
lotta_abb.filter(pl.col("full_word").str.contains(r"\?"))

abbreviation,full_word,full_word_right,translation,note,word_stem,declension,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,note1,note2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Arm.""","""?""","""?""",null,null,null,null,null,"""Arm.""","""Arm.""","""Arm.""","""Arm.""","""Arm.""","""Arm.""","""Arm.""","""Arm.""",null,null
"""cath.""","""catholicus ?""","""catholicus ?""",null,null,"""catholic -i""","""o""",null,null,null,"""cath.""",null,null,null,null,null,null,null
"""compon.""","""componere?""","""componere?""",null,null,"""compon, composu, compost""",null,null,"""compon.""",null,null,null,null,null,null,null,null,null
"""cont.""","""?""","""?""",null,"""jeweils einmal in 1 und 2""",null,null,"""cont.""","""cont.""",null,null,null,null,null,null,null,null,null
"""ed.""","""ediert ?""","""ediert ?""",null,null,null,null,null,null,null,null,"""ed.""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""except.""","""exceptus ?""","""exceptus ?""",null,null,"""except -i""","""o (Adj.)""",null,null,"""except.""",null,null,null,null,null,null,null,null
"""pend.""","""?""","""?""",null,"""siehe lite pend.""",null,null,null,null,null,null,null,null,null,null,null,null,null
"""t. gr.""","""?""","""?""",null,null,null,null,"""t. gr.""","""t. gr.""",null,null,null,null,null,null,null,null,null


In [ ]:
def eq_or_None(abb, alt_abbs):
    comps = []
    for alt_abb in alt_abbs:
        comparison = abb == alt_abb
        comparison.fill_null(True)
        comps.append(comparison)

    return pl.all_horizontal(comps)

# TODO: doesn't work when (all) columns are null - why?
lotta_abb.filter(eq_or_None(pl.col("abbreviation"), [pl.col("RG1"), pl.col("RG2"), pl.col("RG3"), pl.col("RG4"), pl.col("RG5"), pl.col("RG6"), pl.col("RG7"), pl.col("RG8"), pl.col("RG9")]))

abbreviation,full_word,full_word_right,translation,note,word_stem,declension,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,note1,note2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""bb.""","""beati (Plural)""","""beati (Plural)""","""die heiligen""",null,"""beat -orum/arum""","""o/a (Adj.)""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""","""bb.""",null,null
"""ian.""","""ianuarius""","""ianuarius""","""Januar""",null,"""ianuari -i""","""o""","""ian.""","""ian.""","""ian.""","""ian.""","""ian.""","""ian.""","""ian.""","""ian.""","""ian.""",null,null
"""iul.""","""iulius""","""iulius""","""Juli""",null,"""iuli -i""","""o""","""iul.""","""iul.""","""iul.""","""iul.""","""iul.""","""iul.""","""iul.""","""iul.""","""iul.""",null,null
"""leg.""","""leges""","""leges""","""Rechte""",null,"""leg -um (bereits Plural)""","""konsonantisch""","""leg.""","""leg.""","""leg.""","""leg.""","""leg.""","""leg.""","""leg.""","""leg.""","""leg.""",null,null
"""mag.""","""magister""","""magister""","""Magister, Meister, Vorsteher""",null,"""magistr -i""","""o""","""mag.""","""mag.""","""mag.""","""mag.""","""mag.""","""mag.""","""mag.""","""mag.""","""mag.""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""prep.""","""prepositus""","""prepositus""","""Propst, Vorsteher""",null,"""preposit -i ?""","""o ?""","""prep.""","""prep.""","""prep.""","""prep.""","""prep.""","""prep.""","""prep.""","""prep.""","""prep.""",null,null
"""presb.""","""presbiter""","""presbiter""","""Priester""","""auch presbyter""",null,null,"""presb.""","""presb.""","""presb.""","""presb.""","""presb.""","""presb.""","""presb.""","""presb.""","""presb.""",null,null
"""sept.""","""september""","""september""","""September""",null,"""septembr -is""","""konsonantisch""","""sept.""","""sept.""","""sept.""","""sept.""","""sept.""","""sept.""","""sept.""","""sept.""","""sept.""",null,null


In [51]:
lotta_abb.filter(pl.all_horizontal(pl.col("RG1").is_null(), pl.col("RG2").is_null(), pl.col("RG3").is_null(), pl.col("RG4").is_null(), pl.col("RG5").is_null(), pl.col("RG6").is_null(), pl.col("RG7").is_null(), pl.col("RG8").is_null(), pl.col("RG9").is_null()))

abbreviation,full_word,full_word_right,translation,note,word_stem,declension,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,note1,note2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""abb. et. conv.""","""abbas et conventus""","""abbas et conventus""","""Abt und Konvent""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""abbr.""","""litterarum apostolicarum abbre…","""litterarum apostolicarum abbre…","""Abbreviator, Abfasser von Konz…",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""absolv.""","""absolvere ""","""absolvere ""","""lossprechen """,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""accip.""","""accipere""","""accipere""","""empfangen, erhalten""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""adher.""","""adherens; adherentes""","""adherens; adherentes""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tabell. off.""","""tabellionatus officium ?""","""tabellionatus officium ?""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""transfer.""","""transferre""","""transferre""",null,"""siehe transf.""",null,null,null,null,null,null,null,null,null,null,null,null,null
"""Tur.""","""Turonensis""","""Turonensis""","""Turnosen (Münze)""","""siehe T.""",null,null,null,null,null,null,null,null,null,null,null,null,null


In [22]:
import polars as pl
import polars.selectors as cs
alias = pl.read_csv("transformed/aliases.csv")
manual = pl.read_csv("data/manually_revised_abbreviations.csv", row_index_name="index", row_index_offset=1)

In [25]:
siehe = manual.filter(pl.any_horizontal(cs.string().str.contains("siehe", literal=True)))
siehe.join(alias, left_on="index", right_on="source_row_id", how="anti")

index,Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen,Bemerkungen_duplicated_0
u32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
76,"""arg.""","""argentum""","""Silber""",null,"""argent -i""","""o""",null,null,null,"""arg.""","""arg.""","""arg.""","""arg.""","""arg.""","""arg.""",null,"""nicht in Bd. 1, 2, 3; in Bd. 4…"
92,null,"""bulla""","""Bulle""","""v. a. in ""l. b.""; siehe dortig…","""bull -(a)e""","""a""",null,null,null,null,null,null,null,null,null,null,null
181,null,"""communis ""","""gewöhnlich ""","""in Bd. 1 (siehe auch commun.)""",null,null,null,null,null,null,null,null,null,null,null,null,null
194,"""comp.""","""competenter""","""angemessen, befriedigend""","""Bd. 9; siehe compet.""",null,null,null,null,null,null,null,null,null,null,null,null,null
219,null,"""consanguinitas ""","""(Bluts-)verwandtschaft""","""in Bd. 1 (siehe auch consang.)""",null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
962,null,"""?""",null,"""1,2 in t. gr.; siehe t. gr.""",null,null,null,null,null,null,null,null,null,null,null,null,null
975,null,"""testimonialis""",null,"""in Bd. 4; siehe testim.""","""testimonial -is""","""i (Adj.)""",null,null,null,null,null,null,null,null,null,null,null
990,"""translat.""","""translaturus""",null,"""in Bd. 6 laut Abk.-Verz.; sieh…","""translatur -i""","""o (Adj.)""",null,null,null,null,null,null,null,null,null,"""in 6 nicht verwendet; aber vie…",null
